# ✈️ Flight Delay Prediction — Improved Version (100k flights dataset)
Same dataset and idea as `Project_Demo.ipynb`, with these fixes:
1. **Time-based split** (train on older flights, test on newer) instead of a random split
2. **Two models**: *Pre-departure* (no departure delay) and *At-gate* (with departure delay)
3. **More features**: month, weekday, hour, airline, airport and route delay rates (no leakage)
4. **Random Forest vs Gradient Boosting**, plus **threshold tuning** (accuracy alone is misleading: only ~18% of flights are delayed)
5. Saves everything needed for deployment

**Before running:** put `flights_sample_100k.csv` (your file `flights_sample_100k_csv.xls` is a CSV) in Colab, or run the upload cell below.

In [ ]:
import os, json, shutil, warnings
import numpy as np, pandas as pd, joblib
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import KFold
from sklearn.inspection import permutation_importance
from sklearn.metrics import (roc_auc_score, average_precision_score, accuracy_score, roc_curve, precision_recall_curve,
                             precision_score, recall_score, f1_score, confusion_matrix)
warnings.filterwarnings("ignore"); sns.set_theme(style="whitegrid")

DATA_PATH = "flights_sample_100k.csv"     # change if your file has another name, e.g. "flights_sample_100k_csv.xls"
OUT_DIR = "artifacts"; os.makedirs(OUT_DIR, exist_ok=True)
DELAY_MINUTES, SEED = 15, 42

In [ ]:
# Upload cell (Colab only). Skip it if the file is already in the Files panel.
try:
    from google.colab import files
    if not os.path.exists(DATA_PATH):
        up = files.upload(); DATA_PATH = list(up.keys())[0]
except ImportError:
    pass
print("Using:", DATA_PATH)

## 1. Load & clean

In [ ]:
df = pd.read_csv(DATA_PATH)          # works even though the extension says .xls – it is a CSV
print("Raw shape:", df.shape, "| dates:", df.FL_DATE.min(), "→", df.FL_DATE.max())
print("Airlines:", df.AIRLINE.nunique(), "| origin airports:", df.ORIGIN.nunique(), "| cancelled:", f"{df.CANCELLED.mean():.1%}")

df = df.dropna(subset=["ARR_DELAY"]).copy()                  # drops cancelled/diverted flights (no arrival delay)
df["y"] = (df["ARR_DELAY"] > DELAY_MINUTES).astype(int)      # 1 = arrived more than 15 min late
df["date"] = pd.to_datetime(df["FL_DATE"])
df["month"], df["dow"] = df["date"].dt.month, df["date"].dt.dayofweek
df["dep_hour"], df["arr_hour"] = df["CRS_DEP_TIME"] // 100, df["CRS_ARR_TIME"] // 100
df["route"] = df["ORIGIN"] + "_" + df["DEST"]
print(f"Clean rows: {len(df):,} | delayed share: {df.y.mean():.1%}")

# NOTE: DELAY_DUE_*, TAXI_*, WHEELS_*, ELAPSED_TIME, AIR_TIME, ARR_TIME are only known AFTER the flight -> never used (data leakage)

## 2. Quick EDA

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(20, 4))
sns.histplot(df.ARR_DELAY.clip(-60, 240), bins=60, color="purple", ax=ax[0]); ax[0].set_title("Arrival delay (min)")
df.groupby("month").y.mean().plot(kind="bar", ax=ax[1], color="teal"); ax[1].set_title("Delay rate by month")
df.groupby("dep_hour").y.mean().plot(ax=ax[2], marker="o", color="crimson"); ax[2].set_title("Delay rate by departure hour")
df.groupby("AIRLINE").y.mean().sort_values().plot(kind="barh", ax=ax[3], color="orange"); ax[3].set_title("Delay rate by airline")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/eda.png", dpi=120); plt.show()

## 3. Time-based split + features
Train = oldest 70% of dates · Validation = next 10% (chooses the threshold) · Test = newest 20%.
Airport/airline/route delay rates are computed on **training data only** (out-of-fold for train rows) so nothing leaks.

In [ ]:
d70, d80 = df.date.quantile(0.70), df.date.quantile(0.80)
tr, va, te = df[df.date <= d70].copy(), df[(df.date > d70) & (df.date <= d80)].copy(), df[df.date > d80].copy()
for n, f in [("train", tr), ("val", va), ("test", te)]:
    print(f"{n:5s} {len(f):6,} rows  {f.date.min().date()} → {f.date.max().date()}  delayed {f.y.mean():.1%}")

G, M = tr.y.mean(), 20                                   # global rate, smoothing strength
def rate_map(frame, col):
    s = frame.groupby(col).y.agg(["sum", "count"]); return ((s["sum"] + M * G) / (s["count"] + M)).to_dict()

TE = {"ORIGIN": "rate_origin", "DEST": "rate_dest", "AIRLINE_CODE": "rate_airline", "route": "rate_route"}
te_maps = {c: rate_map(tr, c) for c in TE}
for col, new in TE.items():
    oof = pd.Series(np.nan, index=tr.index)
    for a, b in KFold(5, shuffle=True, random_state=SEED).split(tr):
        oof.iloc[b] = tr.iloc[b][col].map(rate_map(tr.iloc[a], col)).values
    tr[new] = oof.fillna(G)
    for f in (va, te): f[new] = f[col].map(te_maps[col]).fillna(G)

airlines = sorted(tr.AIRLINE_CODE.unique()); air_map = {a: i for i, a in enumerate(airlines)}
for f in (tr, va, te): f["airline_id"] = f.AIRLINE_CODE.map(air_map).astype(float)

PRE = ["month", "dow", "dep_hour", "arr_hour", "CRS_ELAPSED_TIME", "DISTANCE", "airline_id", *TE.values()]
MODES = {"A_pre_departure": PRE, "B_at_gate": PRE + ["DEP_DELAY"]}
for k, v in MODES.items(): print(k, "→", v)

## 4. Train & compare (Random Forest = your original model, vs Gradient Boosting)

In [ ]:
def tune_threshold(y, p):
    pr, rc, th = precision_recall_curve(y, p); f1 = 2 * pr * rc / (pr + rc + 1e-9); return float(th[np.argmax(f1[:-1])])

rows, fitted = [], {}
for mode, feats in MODES.items():
    models = {"RandomForest": RandomForestClassifier(n_estimators=200, max_depth=12, min_samples_leaf=20, n_jobs=-1, random_state=SEED),
              "GradientBoosting": HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, max_leaf_nodes=31,
                                   min_samples_leaf=40, l2_regularization=1.0, early_stopping=True, random_state=SEED)}
    for name, m in models.items():
        Xtr = tr[feats].fillna(-1) if name == "RandomForest" else tr[feats]
        fx = (lambda f: f[feats].fillna(-1)) if name == "RandomForest" else (lambda f: f[feats])
        m.fit(Xtr, tr.y)
        pv, pt = m.predict_proba(fx(va))[:, 1], m.predict_proba(fx(te))[:, 1]
        thr = tune_threshold(va.y, pv); yh = (pt >= thr).astype(int)
        rows.append(dict(setup=mode, model=name, ROC_AUC=roc_auc_score(te.y, pt), PR_AUC=average_precision_score(te.y, pt),
                         precision=precision_score(te.y, yh), recall=recall_score(te.y, yh), F1=f1_score(te.y, yh),
                         accuracy=accuracy_score(te.y, yh), threshold=thr))
        fitted[(mode, name)] = (m, feats, thr, pt, name == "RandomForest")
res = pd.DataFrame(rows).round(3)
print(f"Test delayed share = {te.y.mean():.3f} (a model with no skill has PR_AUC ≈ this value)")
display(res); res.to_csv(f"{OUT_DIR}/model_comparison.csv", index=False)

## 5. Plots for the best model of each setup

In [ ]:
best = {m: res[res.setup == m].sort_values("ROC_AUC", ascending=False).iloc[0].model for m in MODES}
fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
for mode in MODES:
    m, feats, thr, pt, _ = fitted[(mode, best[mode])]
    fpr, tpr, _ = roc_curve(te.y, pt); ax[0].plot(fpr, tpr, label=f"{mode} (AUC {roc_auc_score(te.y, pt):.3f})")
ax[0].plot([0, 1], [0, 1], "k--"); ax[0].legend(); ax[0].set_title("ROC curve (test set)")
m, feats, thr, pt, _ = fitted[("B_at_gate", best["B_at_gate"])]
sns.heatmap(confusion_matrix(te.y, pt >= thr), annot=True, fmt="d", cmap="Purples", cbar=False, ax=ax[1],
            xticklabels=["On-time", "Delayed"], yticklabels=["On-time", "Delayed"]); ax[1].set_title("Confusion matrix – B (at-gate)")
m, feats, thr, pt, rf = fitted[("A_pre_departure", best["A_pre_departure"])]
samp = te.sample(min(10_000, len(te)), random_state=SEED)
pi = permutation_importance(m, samp[feats].fillna(-1) if rf else samp[feats], samp.y, scoring="roc_auc", n_repeats=5, random_state=SEED)
pd.Series(pi.importances_mean, index=feats).sort_values().plot(kind="barh", ax=ax[2], color="skyblue"); ax[2].set_title("What drives pre-departure delays")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/evaluation.png", dpi=120); plt.show()

## 6. Save for deployment

In [ ]:
bundle = {mode: dict(model=fitted[(mode, best[mode])][0], features=MODES[mode], threshold=fitted[(mode, best[mode])][2],
                     model_name=best[mode], fill_na=-1 if best[mode] == "RandomForest" else None) for mode in MODES}
bundle["meta"] = dict(te_maps=te_maps, te_cols=TE, global_rate=G, airline_map=air_map, delay_minutes=DELAY_MINUTES)
joblib.dump(bundle, f"{OUT_DIR}/flight_delay_bundle.joblib", compress=3)

# lookup tables for the web app: per-route typical distance / duration / hour, airline names, airport names
route_info = df.groupby("route").agg(n=("y", "size"), distance=("DISTANCE", "median"), duration=("CRS_ELAPSED_TIME", "median"),
                                      delay_rate=("y", "mean")).reset_index()
route_info.to_csv(f"{OUT_DIR}/routes.csv", index=False)
df[["AIRLINE", "AIRLINE_CODE"]].drop_duplicates().to_csv(f"{OUT_DIR}/airlines.csv", index=False)
pd.concat([df[["ORIGIN", "ORIGIN_CITY"]].set_axis(["code", "city"], axis=1),
           df[["DEST", "DEST_CITY"]].set_axis(["code", "city"], axis=1)]).drop_duplicates("code").to_csv(f"{OUT_DIR}/airports.csv", index=False)
print(sorted(os.listdir(OUT_DIR)))
shutil.make_archive("flight_delay_artifacts", "zip", OUT_DIR)
try:
    from google.colab import files; files.download("flight_delay_artifacts.zip")
except ImportError:
    pass
print("✅ Done. Send me the model_comparison table and we'll deploy the final model.")